# Trimming Strategies [Step 01.03]

> **MLCourse - Agentic AI - Agent Patterns**

You are over budget. Something has to go. **Which something?**

Three strategies, three different theories of what matters:

```
  RECENCY    keep the newest       "the last thing said is the relevant thing"
  RELEVANCE  keep the most similar "keep whatever matches the current question"
  PRIORITY   keep by declared rank "some things must never be dropped"
```

They disagree, and the disagreement is the interesting part. In this notebook we
build a conversation where **each strategy loses a different critical fact**, and
then measure which one survives a question that needs that fact.

### What you'll learn

- Implementations of all three strategies against one shared budget.
- The specific failure mode of each.
- Why production systems use a **hybrid**, and what the hybrid looks like.

### Why it matters

Recency is the default in almost every framework, and it is wrong in a
predictable way: the constraint the user stated in turn 2 ("I'm vegetarian",
"budget is 400 EUR", "must be wheelchair accessible") falls off the front of the
window and the agent cheerfully violates it in turn 12. Users experience this as
the agent "forgetting", and it is the number one complaint about long-running
chat agents.

### Prerequisites

- [02_token_budget](02_token_budget.ipynb)
- [03_rag_advanced/11_reranking](../../03_rag_advanced/11_reranking) - relevance scoring.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. A conversation with a landmine in it

The setup: a trip-planning conversation. **Turn 2 contains a hard constraint**
(a peanut allergy). Then a long stretch of chit-chat pushes it far into the past.
Finally the user asks a question whose correct answer depends on that old turn.

This is not a contrived example - it is the shape of nearly every real complaint
about agent memory.

In [2]:
CONVERSATION = [
    ("user",      "Hi! I'm planning a week in Lisbon in October."),
    ("assistant", "Lovely choice. What would you like help with first?"),
    # ---- the landmine: turn index 2 -------------------------------------
    ("user",      "One important thing up front: I have a severe peanut allergy, "
                  "so nothing you suggest can involve peanuts or peanut oil."),
    ("assistant", "Noted - I'll keep that in mind for every food suggestion."),
    ("user",      "What's the weather like in October?"),
    ("assistant", "Mild and pleasant: highs around 22C, lows around 15C, with "
                  "occasional rain in the second half of the month."),
    ("user",      "Do I need an adapter for my plug?"),
    ("assistant", "Portugal uses Type F sockets at 230V, so a European adapter."),
    ("user",      "Is the metro easy to use?"),
    ("assistant", "Very - four lines, and a Viva Viagem card covers metro, tram and bus."),
    ("user",      "How far is Sintra?"),
    ("assistant", "About 40 minutes by train from Rossio station, runs twice hourly."),
    ("user",      "Any good day trips besides Sintra?"),
    ("assistant", "Cascais for the coast, Setubal for wine, Evora for Roman ruins."),
    ("user",      "Should I rent a car?"),
    ("assistant", "Not for the city - parking is hard. Only worth it for Alentejo."),
    ("user",      "What about tipping?"),
    ("assistant", "Not obligatory. Rounding up or 5-10% for good service is plenty."),
]

FINAL_QUESTION = "Recommend one traditional Portuguese dessert I should try, and say why it's safe for me."

print("turns:", len(CONVERSATION))
print("total history tokens:",
      approx_tokens("\n".join("%s: %s" % t for t in CONVERSATION)))
print("\nthe constraint lives at turn index 2:")
print(" ", CONVERSATION[2][1][:70], "...")

turns: 18
total history tokens: 275

the constraint lives at turn index 2:
  One important thing up front: I have a severe peanut allergy, so nothi ...


### 2. The budget, and the three strategies

We set a history budget of **160 tokens**, which fits roughly four turns out of
sixteen. All three strategies get the same budget. Only the selection rule differs.

In [3]:
HISTORY_BUDGET = 160


def turn_tokens(turn):
    return approx_tokens("%s: %s" % turn)


def pack(turns, budget):
    """Greedily take turns in the given order until the budget is exhausted."""
    kept, used = [], 0
    for t in turns:
        c = turn_tokens(t)
        if used + c > budget:
            continue                 # skip, but keep trying smaller later ones
        kept.append(t)
        used += c
    return kept, used


# --- Strategy A: RECENCY ------------------------------------------------------
def trim_recency(turns, budget):
    """Keep the newest turns. The framework default, everywhere."""
    kept, used = pack(list(reversed(turns)), budget)
    return list(reversed(kept)), used

### Strategy B: RELEVANCE


In [ ]:
# Score every turn against the current question, keep the best scorers.
# We use a local embedding model (fastembed, ONNX - no GPU, no torch, no network
# call per query) so this is fast and deterministic.
import numpy as np
from fastembed import TextEmbedding

_EMB = TextEmbedding("BAAI/bge-small-en-v1.5")


def embed(texts):
    v = np.array(list(_EMB.embed(list(texts))), dtype="float32")
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def trim_relevance(turns, budget, query):
    """Keep the turns most semantically similar to the current question."""
    texts = ["%s: %s" % t for t in turns]
    sims = embed(texts) @ embed([query])[0]
    order = sorted(range(len(turns)), key=lambda i: -sims[i])
    kept_idx, used = [], 0
    for i in order:
        c = turn_tokens(turns[i])
        if used + c > budget:
            continue
        kept_idx.append(i)
        used += c
    kept_idx.sort()                              # restore chronological order
    return [turns[i] for i in kept_idx], used, sims


### Strategy C: PRIORITY


In [ ]:
# Turns carry an explicit tier. Tier 1 is pinned: it is NEVER dropped, whatever
# the budget. Everything else competes for the remainder by recency.
#
# The tiering here is done by a simple keyword rule so the notebook stays fast
# and deterministic. In production this is usually a cheap classifier call, and
# it runs ONCE per turn (not once per request), so it is affordable.

PIN_MARKERS = ("allerg", "must ", "never ", "budget is", "i cannot", "i can't",
               "important thing", "wheelchair", "vegetarian", "vegan")


def tier(turn):
    role, text = turn
    if role == "user" and any(m in text.lower() for m in PIN_MARKERS):
        return 1                      # a durable constraint: pin it
    return 2                          # ordinary chatter


def trim_priority(turns, budget):
    pinned = [t for t in turns if tier(t) == 1]
    used = sum(turn_tokens(t) for t in pinned)
    rest, extra = pack([t for t in turns if tier(t) == 2][::-1], budget - used)
    kept = pinned + list(reversed(rest))
    kept.sort(key=lambda t: turns.index(t))
    return kept, used + extra


for t in CONVERSATION:
    if tier(t) == 1:
        print("PINNED:", t[1][:80])


### 3. Run all three on the same budget

Now the comparison. Watch specifically whether the peanut-allergy turn survives.

In [6]:
rec, rec_used = trim_recency(CONVERSATION, HISTORY_BUDGET)
rel, rel_used, sims = trim_relevance(CONVERSATION, HISTORY_BUDGET, FINAL_QUESTION)
pri, pri_used = trim_priority(CONVERSATION, HISTORY_BUDGET)


def has_constraint(kept):
    return any("peanut" in t[1].lower() for t in kept)


print("%-11s %6s %7s %8s   %s" % ("strategy", "turns", "tokens", "allergy?", "kept turn indexes"))
print("-" * 78)
for name, kept, used in (("recency", rec, rec_used),
                         ("relevance", rel, rel_used),
                         ("priority", pri, pri_used)):
    idx = [CONVERSATION.index(t) for t in kept]
    print("%-11s %6d %7d %8s   %s"
          % (name, len(kept), used, "KEPT" if has_constraint(kept) else "LOST", idx))

strategy     turns  tokens allergy?   kept turn indexes
------------------------------------------------------------------------------
recency         11     156     LOST   [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
relevance       12     158     KEPT   [0, 1, 2, 3, 4, 7, 8, 10, 12, 13, 14, 16]
priority        10     156     KEPT   [2, 9, 10, 11, 12, 13, 14, 15, 16, 17]


### Why did relevance behave the way it did? Look at the similarity scores.


In [ ]:
print("similarity of each turn to the final question:")
for i, (t, s) in enumerate(zip(CONVERSATION, sims)):
    mark = "  <-- the constraint" if i == 2 else ""
    print("  %2d  %.3f  %s: %s%s" % (i, s, t[0][:4], t[1][:52], mark))


### Why each strategy fails

- **Recency** loses the constraint because the constraint is *old*. Recency has
  no concept of importance - it only knows about time.
- **Relevance** can go either way, and *why* is the lesson: it scores the turn
  about the allergy against a question about *desserts*. If the wording does not
  overlap, a critical constraint scores low and is discarded. Relevance
  optimises for topical match, and constraints are frequently off-topic.
- **Priority** keeps the constraint by construction - but it needs a rule that
  recognises a constraint when it sees one, and a bad rule pins junk (which
  quietly eats the whole budget).

### 4. Does it change the answer?

Assertions are cheap. Send all three trimmed contexts and read the replies.

In [8]:
SYS = ("You are a travel assistant. Use the conversation so far. "
       "Respect any constraints the traveller has stated. Two sentences maximum.")

results = {}
for name, kept in (("recency", rec), ("relevance", rel), ("priority", pri)):
    msgs = [("system", SYS)] + list(kept) + [("user", FINAL_QUESTION)]
    out = chat(msgs, max_tokens=140)
    results[name] = out
    print("=" * 74)
    print("%s   (input tokens: %d)" % (name.upper(), out.usage_metadata["input_tokens"]))
    print(out.content)

RECENCY   (input tokens: 281)
Try Pastel de Nata, a flaky custard tart that is naturally gluten-free. It's a classic Lisbon treat that fits your dietary needs perfectly.


RELEVANCE   (input tokens: 309)
Try Pastéis de Nata, the classic custard tarts, as they are made with just egg, milk, and sugar. This ensures they are naturally free from peanuts and peanut oil.


PRIORITY   (input tokens: 266)
Try Pastel de Nata, the classic egg custard tart. It is made with flour, eggs, and sugar, so it contains no peanuts or peanut oil.


In [9]:
# A crude but honest automatic check: does the reply show awareness of the allergy?
print("%-11s %10s %14s" % ("strategy", "allergy in", "mentions allergy"))
print("%-11s %10s %14s" % ("", "context", "in the reply"))
print("-" * 40)
for name, kept in (("recency", rec), ("relevance", rel), ("priority", pri)):
    aware = any(w in results[name].content.lower() for w in ("peanut", "allerg", "nut"))
    print("%-11s %10s %14s" % (name, has_constraint(kept), aware))

strategy    allergy in mentions allergy
               context   in the reply
----------------------------------------
recency          False          False
relevance         True           True
priority          True           True


> **Read this honestly.** A model can produce a *safe-sounding* dessert answer
> without knowing about the allergy at all - most Portuguese desserts are
> peanut-free by chance. "Did not poison the user this time" is not the same as
> "knew about the constraint". That is exactly why the table above reports the
> two columns separately: what was in the context, and what the reply
> demonstrates. Only the first is under your control.

### 5. The hybrid, which is what you should actually ship

Nobody runs pure recency in production for a stateful agent. The standard
arrangement is three tiers, filled in this order:

```
  1. PINNED       constraints, identity, task goal      never dropped
  2. RECENT       the last k turns verbatim             tail of the conversation
  3. RELEVANT     older turns retrieved by similarity   fills the remainder
```

Pinning first is what makes it safe; recency second is what keeps it coherent;
relevance last is what makes the leftover budget useful.

In [10]:
def trim_hybrid(turns, budget, query, keep_recent=2):
    """Pinned first, then the last `keep_recent` turns, then relevance for the rest."""
    pinned = [t for t in turns if tier(t) == 1]
    recent = [t for t in turns[-keep_recent * 2:] if t not in pinned]

    kept = pinned + recent
    used = sum(turn_tokens(t) for t in kept)

    remaining = [t for t in turns if t not in kept]
    if remaining:
        texts = ["%s: %s" % t for t in remaining]
        sims2 = embed(texts) @ embed([query])[0]
        for i in sorted(range(len(remaining)), key=lambda i: -sims2[i]):
            c = turn_tokens(remaining[i])
            if used + c <= budget:
                kept.append(remaining[i])
                used += c

    kept.sort(key=lambda t: turns.index(t))
    return kept, used


hyb, hyb_used = trim_hybrid(CONVERSATION, HISTORY_BUDGET, FINAL_QUESTION)
print("hybrid kept turn indexes:", [CONVERSATION.index(t) for t in hyb])
print("tokens used             : %d / %d" % (hyb_used, HISTORY_BUDGET))
print("constraint kept         :", has_constraint(hyb))
print()
out = chat([("system", SYS)] + list(hyb) + [("user", FINAL_QUESTION)], max_tokens=140)
print(out.content)

hybrid kept turn indexes: [0, 1, 2, 3, 12, 13, 14, 15, 16, 17]
tokens used             : 153 / 160
constraint kept         : True



Try Pastel de Nata, the classic egg custard tart. It's made with flour, eggs, and sugar, so it's naturally peanut-free.


### 6. Pitfalls

- **Trimming the system prompt.** It is small and it is your only control
  surface. Never include it in the trimming pool.
- **Breaking tool-call pairs.** If a turn is an assistant tool call, its matching
  tool *result* must stay with it. Dropping one and keeping the other produces
  API errors or hallucinated results. Always trim in whole exchanges.
- **Trusting keyword pinning.** The `PIN_MARKERS` list here is a teaching
  device. Real systems use a classifier and *review the pins*, because an
  over-eager pinner will fill the entire budget with tier 1.
- **Trimming instead of summarising.** Dropping a turn destroys it permanently.
  Module 02 covers the alternative: compress it and keep the information.

### Recap

| Strategy | Keeps | Fails when |
|---|---|---|
| Recency | The newest turns | The critical fact is old |
| Relevance | The most similar turns | The critical fact is off-topic |
| Priority | Whatever you declared important | Your importance rule is wrong |
| Hybrid | Pinned, then recent, then relevant | (the sane default) |

**Next:** [04_lost_in_the_middle](04_lost_in_the_middle.ipynb) - even the context
that survives trimming is not read evenly. We measure it.